In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset , DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt

In [8]:
# check for gpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [9]:
import kagglehub
path = kagglehub.dataset_download("zalando-research/fashionmnist")

train_path = "/kaggle/input/fashionmnist/fashion-mnist_train.csv"
df = pd.read_csv(train_path)

Using Colab cache for faster access to the 'fashionmnist' dataset.


In [12]:
df.sample()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,pixel11,pixel12,pixel13,pixel14,pixel15,pixel16,pixel17,pixel18,pixel19,pixel20,pixel21,pixel22,pixel23,pixel24,pixel25,pixel26,pixel27,pixel28,pixel29,pixel30,pixel31,pixel32,pixel33,pixel34,pixel35,pixel36,pixel37,pixel38,pixel39,...,pixel745,pixel746,pixel747,pixel748,pixel749,pixel750,pixel751,pixel752,pixel753,pixel754,pixel755,pixel756,pixel757,pixel758,pixel759,pixel760,pixel761,pixel762,pixel763,pixel764,pixel765,pixel766,pixel767,pixel768,pixel769,pixel770,pixel771,pixel772,pixel773,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
13152,2,0,0,0,0,0,0,0,0,0,98,180,151,120,112,115,149,205,192,74,3,0,0,0,1,0,0,0,0,0,0,0,1,0,0,27,117,192,183,192,...,201,212,215,206,143,30,128,120,21,0,0,0,0,0,0,0,0,0,0,15,101,118,120,122,124,124,126,130,135,141,145,158,72,0,0,0,0,0,0,0


In [11]:
# train test split
X = df.drop('label', axis=1)
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
# scaling
X_train = X_train / 255.0
X_test = X_test / 255.0

In [14]:
# dataset class
class CustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features = torch.tensor(features.values,dtype=torch.float32).reshape(-1,1,28,28)
    self.lables = torch.tensor(labels.values,dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self,idx):
    x = self.features[idx]
    y = self.lables[idx]
    return x,y

In [15]:
train_dataset = CustomDataset(X_train,y_train)
test_dataset = CustomDataset(X_test,y_test)

In [16]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=True)

In [17]:
# define CNN class
class CNN(nn.Module):

  def __init__(self,input_features):

    super().__init__()

    # features extraction
    self.features = nn.Sequential(
        nn.Conv2d(in_channels=input_features,out_channels=32,kernel_size=3,stride=1,padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(kernel_size=2,stride=2),

        nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3,stride=1,padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(kernel_size=2,stride=2)
    )

    # classification
    self.classifier = nn.Sequential(
        nn.Flatten(),

        nn.Linear(in_features=64*7*7,out_features=128),
        nn.ReLU(),
        nn.Dropout(0.2),

        nn.Linear(in_features=128,out_features=64),
        nn.ReLU(),
        nn.Dropout(0.2),

        nn.Linear(in_features=64,out_features=10)
    )

  def forward(self,x):

    feature_output = self.features(x)
    classifier_output = self.classifier(feature_output)

    return classifier_output


In [21]:
  model = CNN(1)
  model.to(device)


# optimizer selection
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.SGD(model.parameters(),lr=0.01,weight_decay=1e-4)


In [22]:
# training loop

for epoch in range(25):

  total_loss = 0

  for batch_features,batch_labels in train_loader:
    # move to gpu
    batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)

    # forward pass
    predications = model(batch_features)

    # calculate loss
    loss = criterion(predications,batch_labels)

    # make grad zero
    optimizer.zero_grad()

    # backward pass
    loss.backward()

    # update parameter
    optimizer.step()

    total_loss += loss.item()


  print(f'epoch: {epoch+1} avg_loss: {total_loss/len(train_loader)}')

epoch: 1 avg_loss: 0.5348712266385556
epoch: 2 avg_loss: 0.3247819577579697
epoch: 3 avg_loss: 0.2756336233442028
epoch: 4 avg_loss: 0.2452352785989642
epoch: 5 avg_loss: 0.21775503582134842
epoch: 6 avg_loss: 0.20265684374173482
epoch: 7 avg_loss: 0.1860946516431868
epoch: 8 avg_loss: 0.1709636859583358
epoch: 9 avg_loss: 0.1592238039827595
epoch: 10 avg_loss: 0.14374519833518812
epoch: 11 avg_loss: 0.1310424282409561
epoch: 12 avg_loss: 0.12074354442457358
epoch: 13 avg_loss: 0.11395809688046575
epoch: 14 avg_loss: 0.1024481926675265
epoch: 15 avg_loss: 0.09285327288865422
epoch: 16 avg_loss: 0.08581505937657008
epoch: 17 avg_loss: 0.0800766871892071
epoch: 18 avg_loss: 0.07292954074606921
epoch: 19 avg_loss: 0.06676470505376346
epoch: 20 avg_loss: 0.06273258529617064
epoch: 21 avg_loss: 0.060436437456092486
epoch: 22 avg_loss: 0.05228411384885354
epoch: 23 avg_loss: 0.04798511414885676
epoch: 24 avg_loss: 0.04511291794046216
epoch: 25 avg_loss: 0.04100775200113033


In [24]:
 # set model to eval mode
model.eval()

total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_loader:
    batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)


0.9184166666666667
